In [1]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")


OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AI


In [3]:
# Connect to client libraries

openai = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [4]:
OPENAI_MODEL = "gpt-5"
GEMINI_MODEL = "gemini-2.5-pro"
OPENAI_MODEL2 = "gpt-5-nano"
GEMINI_MODEL2 = "gemini-2.5-flash-lite"

In [5]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '10',
  'version': '10.0.19045',
  'kernel': '10',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': ''},
 'package_managers': ['winget'],
 'cpu': {'brand': 'Intel(R) Core(TM) i5-8300H CPU @ 2.30GHz',
  'cores_logical': 8,
  'cores_physical': 4,
  'simd': []},
 'toolchain': {'compilers': {'gcc': '', 'g++': '', 'clang': '', 'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

In [6]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = openai.chat.completions.create(model=OPENAI_MODEL, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))
    

Short answer: you do not have a C++ compiler installed. You’ll need to install one.

Simplest way on your Windows 10 system (using winget): install Microsoft Visual Studio 2022 Build Tools with the C++ workload.

Step-by-step
1) Open an elevated PowerShell (Run as Administrator).
2) Run this one command (silent, minimal UI, includes recommended C++ tools and Windows SDK):
   winget install -e --id Microsoft.VisualStudio.2022.BuildTools --override "--quiet --wait --norestart --add Microsoft.VisualStudio.Workload.VCTools --includeRecommended"
3) When it finishes, you’re ready to compile C++.

Python commands to compile and run main.cpp (optimized for fastest runtime)
- This uses MSVC with high optimization, link-time optimization, and AVX2 for your CPU.

Use these exactly:

compile_command = [
    "cmd.exe", "/d", "/s", "/c",
    "\"%ProgramFiles(x86)%\\Microsoft Visual Studio\\2022\\BuildTools\\Common7\\Tools\\VsDevCmd.bat\" -arch=x64 -host_arch=x64 && cl /nologo /O2 /EHsc /std:c++20 /arch:AVX2 /GL /DNDEBUG /Fe:main.exe main.cpp /link /LTCG"
]

run_command = ["main.exe"]

Notes
- Run your Python script from the directory that contains main.cpp.
- The command above initializes the MSVC build environment and then compiles and links in one step for best runtime performance.

In [8]:
compile_command = [ "cmd.exe", "/d", "/s", "/c", '''%ProgramFiles(x86)%\Microsoft Visual Studio\2022\BuildTools\Common7\Tools\VsDevCmd.bat" -arch=x64 -host_arch=x64 && cl /nologo /O2 /EHsc /std:c++20 /arch:AVX2 /GL /DNDEBUG /Fe:main.exe main.cpp /link /LTCG''' ]

run_command = ["main.exe"]

<>:1: SyntaxWarning: invalid escape sequence '\M'
<>:1: SyntaxWarning: invalid escape sequence '\M'
C:\Users\asus\AppData\Local\Temp\ipykernel_15872\4115401377.py:1: SyntaxWarning: invalid escape sequence '\M'
  compile_command = [ "cmd.exe", "/d", "/s", "/c", '''%ProgramFiles(x86)%\Microsoft Visual Studio\2022\BuildTools\Common7\Tools\VsDevCmd.bat" -arch=x64 -host_arch=x64 && cl /nologo /O2 /EHsc /std:c++20 /arch:AVX2 /GL /DNDEBUG /Fe:main.exe main.cpp /link /LTCG''' ]


In [9]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [10]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [11]:
def write_output(cpp):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp)

In [12]:
def port(client, model, python):
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)

In [13]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [14]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [15]:
run_python(pi)

Result: 3.141592656089
Execution Time: 72.549029 seconds


In [16]:
port(openai, OPENAI_MODEL2, pi)

In [22]:
compile_command = [
    "cmd.exe",
    "/d",
    "/s",
    "/c",
    r'"%ProgramFiles(x86)%\Microsoft Visual Studio\2022\BuildTools\Common7\Tools\VsDevCmd.bat" -arch=x64 -host_arch=x64 && cl /nologo /O2 /EHsc /std:c++20 /arch:AVX2 /GL /DNDEBUG /Fe:"e:\\Projects\\ai_engineer\\learn\\week4\\main.exe" "e:\\Projects\\ai_engineer\\learn\\week4\\main.cpp" /link /LTCG'
]
run_command = [r"e:\Projects\ai_engineer\learn\week4\main.exe"]

In [23]:
# Use the commands from GPT 5

def compile_and_run():
    subprocess.run(compile_command, check=True, text=True, capture_output=True)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)

In [24]:
compile_and_run()

CalledProcessError: Command '['cmd.exe', '/d', '/s', '/c', '"%ProgramFiles(x86)%\\Microsoft Visual Studio\\2022\\BuildTools\\Common7\\Tools\\VsDevCmd.bat" -arch=x64 -host_arch=x64 && cl /nologo /O2 /EHsc /std:c++20 /arch:AVX2 /GL /DNDEBUG /Fe:"e:\\\\Projects\\\\ai_engineer\\\\learn\\\\week4\\\\main.exe" "e:\\\\Projects\\\\ai_engineer\\\\learn\\\\week4\\\\main.cpp" /link /LTCG']' returned non-zero exit status 1.

In [ ]:
port(openai, OPENAI_MODEL, pi)
compile_and_run()

In [25]:
port(gemini, GEMINI_MODEL, pi)
compile_and_run()


RateLimitError: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro\nPlease retry in 33.205119192s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{

In [27]:
port(gemini, GEMINI_MODEL2, pi)
# compile_and_run()

In [ ]:
print(f"""
In Ed's experiments, the performance speedups were:

4th place: Claude Sonnet 4.5: {83.489273/0.104241:.0f}X speedup
3rd place: GPT-5: {83.489273/0.082168:.0f}X speedup
2nd place: Grok 4: {83.489273/0.018092:.0f}X speedup
1st place: Gemini 2.5 Pro: {83.489273/0.013314:.0f}X speedup
""")